# Water consumption in the Canary Islands (1996-2022)

**Dataset**: Volumen y volumen por habitante y dia de agua segun ciclos del agua. Canarias por anos

**Source**: [ISTAC](https://datos.canarias.es) - biennial data on water volume and per capita consumption by water cycle in the Canary Islands

**Questions**: How has water consumption evolved? What share goes to households vs losses? How significant are real losses in a territory dependent on desalination?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8-whitegrid")
OUTPUT_DIR = Path("output/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = "data/raw/volumen-y-volumen-por-habitante-y-dia-de-agua-segun-ciclos-del-agua-canarias-por-anos.csv"
df = pd.read_csv(CSV_PATH)
print("Rows:", len(df), "| Columns:", len(df.columns))
df = df.rename(columns={
    "CICLO_AGUA#es": "water_cycle",
    "CICLO_AGUA_CODE": "cycle_code",
    "MEDIDAS#es": "measure",
    "MEDIDAS_CODE": "measure_code",
    "TIME_PERIOD_CODE": "year",
    "OBS_VALUE": "value"
})
df["year"] = df["year"].astype(int)
print("Time range:", int(df["year"].min()), "->", int(df["year"].max()))
print("Water cycles:", sorted(df["water_cycle"].unique()))
df.head(2)

In [ ]:
# Volume and per-capita subsets
vol = df[df["measure"] == "Volumen"].copy()
per_capita = df[df["measure"] == "Volumen por habitante y d\u00eda"].copy()
print("Volume years with data:", sorted(vol[vol["value"].notna()]["year"].unique()))
print("Per capita years with data:", sorted(per_capita[per_capita["value"].notna()]["year"].unique()))

In [ ]:
# 1. Water volume by cycle over time
CYCLES_VOL = [
    ("Agua tratada suministrada", "Treated water supplied", "#2E86AB"),
    ("Hogares", "Households", "#E94F37"),
    ("Consumos municipales", "Municipal consumption", "#F39C12"),
    ("Sectores econ\u00f3micos", "Economic sectors", "#28A745"),
]
fig, ax = plt.subplots(figsize=(12, 5))
for cycle, label, color in CYCLES_VOL:
    d = vol[(vol["water_cycle"] == cycle) & vol["value"].notna()]
    ax.plot(d["year"], d["value"], label=label, color=color, linewidth=2, marker="o", markersize=4)
ax.set_xlabel("Year")
ax.set_ylabel("Volume (hm\u00b3)")
ax.set_title("Water volume by cycle - Canary Islands (biennial)")
ax.legend(fontsize=8, loc="upper right")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "water_volume_by_cycle.png", dpi=150)
plt.show()
print("Saved: output/figures/water_volume_by_cycle.png")

In [ ]:
# 2. Per capita consumption trend
fig, ax = plt.subplots(figsize=(12, 5))
for cycle, label, color in [
    ("Agua tratada suministrada", "Treated water supplied", "#2E86AB"),
    ("Hogares", "Household consumption", "#E94F37"),
]:
    d = per_capita[(per_capita["water_cycle"] == cycle) & per_capita["value"].notna()]
    if len(d) > 0:
        ax.plot(d["year"], d["value"], label=label, color=color, linewidth=2, marker="o", markersize=4)
ax.set_xlabel("Year")
ax.set_ylabel("Litres per inhabitant per day")
ax.set_title("Per capita water consumption - Canary Islands (biennial)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "water_per_capita_trend.png", dpi=150)
plt.show()
print("Saved: output/figures/water_per_capita_trend.png")

In [ ]:
# 3. Water losses - real and apparent
real_loss = vol[(vol["water_cycle"] == "P\u00e9rdidas reales") & vol["value"].notna()].copy()
apparent_loss = vol[(vol["water_cycle"] == "P\u00e9rdidas aparentes") & vol["value"].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(real_loss["year"], real_loss["value"], color="#E94F37", linewidth=2, marker="o", markersize=4, label="Real losses")
axes[0].set_title("Real water losses - Canary Islands")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("hm\u00b3")
axes[0].legend(fontsize=8)

axes[1].plot(apparent_loss["year"], apparent_loss["value"], color="#F39C12", linewidth=2, marker="s", markersize=4, label="Apparent losses")
axes[1].set_title("Apparent water losses - Canary Islands")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("hm\u00b3")
axes[1].legend(fontsize=8)

fig.suptitle("Water losses - Canary Islands (2007-2022, biennial)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "water_losses_analysis.png", dpi=150)
plt.show()
print("Saved: output/figures/water_losses_analysis.png")

In [ ]:
# 4. Key numbers
print("=== Key numbers (2022) ===")
for cycle, label in [
    ("Agua tratada suministrada", "Treated water supplied"),
    ("P\u00e9rdidas reales", "Real losses"),
    ("P\u00e9rdidas aparentes", "Apparent losses"),
    ("Hogares", "Households"),
]:
    d = vol[(vol["water_cycle"] == cycle) & vol["value"].notna()].sort_values("year").tail(1)
    if len(d) > 0:
        row = d.iloc[0]
        print(f"{label} ({int(row['year'])}): {round(row['value'], 1)} hm\u00b3")

print("\nPer capita treated water supplied (L/inhabitant/day):")
pc = per_capita[(per_capita["water_cycle"] == "Agua tratada suministrad") & per_capita["value"].notna()].sort_values("year")
for _, r in pc.iterrows():
    print(f"  {int(r['year'])}: {round(r['value'], 1)}")